In [ ]:
import sys
import numpy as np
import importlib
import dragons_chime_conv_combine
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u
from reproject import reproject_interp
from reproject import reproject_from_healpix, reproject_to_healpix

## Read in 'raw' CHIME ringmap for selected frequency

In [ ]:
#################
fpick = 408
#################

hdu = fits.open('/srv/data/chime/XX_raw.hpx.fits')
hdr = hdu[0].header
wcs = WCS(hdr)

freqs = wcs.all_pix2world(0,range(wcs.array_shape[0]),0)[1]/1e6

fidx = abs(freqs-fpick).argmin()
print(freqs[fidx])

map = hdu[0].data[fidx]
print(map.shape)

## Make new header and wcs for plate caree projection, make RA and dec arrays

In [ ]:
dxy = 0.125

hdu_new = fits.PrimaryHDU()
hdr_2D = hdu_new.header
hdr_2D['NAXIS']  = 2
hdr_2D['NAXIS1'] = 2880
hdr_2D['NAXIS2'] = 1440
hdr_2D['COORDSYS'] = 'icrs'
hdr_2D['CRPIX1'] = 1440.5
hdr_2D['CRPIX2'] = 720.5
hdr_2D['CRVAL1'] = 180.0
hdr_2D['CRVAL2'] = 0.0
hdr_2D['CDELT1'] = -dxy
hdr_2D['CDELT2'] = dxy
hdr_2D['CUNIT1'] = 'deg  '
hdr_2D['CUNIT2'] = 'deg  '
hdr_2D['CTYPE1'] = 'RA---CAR'
hdr_2D['CTYPE2'] = 'DEC--CAR'

wcs2 = WCS(hdr_2D)
decs = wcs2.all_pix2world(0,range(wcs2.array_shape[0]),0)[1]
RAs  = wcs2.all_pix2world(range(wcs2.array_shape[1]),0,0)[0]

print(decs)
print(RAs)



## Reproject from healpix to plate caree

In [ ]:
map_new_full, footprint = reproject_from_healpix((map, 'icrs'),hdr_2D, nested=False)

map_new_full[np.isnan(map_new_full)] = 0.0

plt.imshow(map_new_full,origin='lower',vmin=-20,vmax=20,cmap='RdBu_r')

## Functions to map subsets and their FFTs

In [ ]:
def get_uv_axes(image, dxy, freq):
    
    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy

    uv_m = uv_freq_m*(3e8/(freq*1e6))*180/np.pi
    uv_n = uv_freq_n*(3e8/(freq*1e6))*180/np.pi

    #xuv, yuv = np.meshgrid(uv_m, uv_n)
    #ruv = np.sqrt(xuv**2+yuv**2)
    
    #b = np.pi/(Rlim[1]-Rlim[0])
    #r0 = (Rlim[0]+Rlim[1])/2.

    return uv_m, uv_n

In [ ]:
def make_map_and_FFT(dec, RA, freq, wcs, RA_extent=50, dec_extent=20):

    print(dec, freq)

    # RA and dec limit indices:
    RAmin = RA - RA_extent/2
    RAmax = RA + RA_extent/2
    decmin = dec - dec_extent/2
    decmax = dec + dec_extent/2
    idxRA  = np.where( (RAs > RAmin) & (RAs < RAmax) )[0]
    idxdec = np.where( (decs > decmin) & (decs < decmax) )[0]

    # Make subset of map for FFT:
    map_sub = map_new_full[idxdec][:,idxRA]
    print(map_sub.shape)

    # Calculate FFT:
    image_FFT = np.fft.fft2(map_sub)
    image_FFT_shift = np.fft.fftshift(image_FFT)
    uv_m, uv_n = get_uv_axes(map_sub, dxy, freq)

    # Make the plots:
    fig = plt.figure(figsize=(18,6))
    plt.subplots_adjust(left=0.05,right=0.95,top=0.95,bottom=0.1,wspace=-0.3)

    # The map:
    ax1 = fig.add_subplot(121, projection=wcs.celestial)
    crange = SkyCoord([RAmax,RAmin], [decmin,decmax], frame=ICRS, unit=(u.deg, u.deg))
    ax1.imshow(map_new_full,origin='lower',vmin=-20,vmax=20,cmap='RdBu_r')
    ax1.set_ylim(wcs.world_to_pixel(crange)[1])
    ax1.set_xlim(wcs.world_to_pixel(crange)[0])
    ax1.set_xlabel('RA')
    ax1.set_ylabel('dec')
    ax1.set_title('central dec = '+str(dec)+r'$^{\circ}$')

    # The FFT:
    ax2 = fig.add_subplot(144)
    ax2.imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=30000,cmap='viridis',
                  extent=(uv_m[0],uv_m[-1],uv_n[0],uv_n[-1]))
    ax2.set_xlabel('u (m)')
    ax2.set_ylabel('v (m)')
    ax2.set_xlim(-120,120)
    ax2.set_ylim(-90,90)
    for x in [22,44,66]:
        ax2.axvline(x=x,color='red',linestyle='dashed')
    
    plt.savefig('../PLOTS/uv_dec_'+str(dec)+'_full.png')

    return

## Make the maps and FFTs

In [ ]:
dec_list = np.arange(10,71,1)
print(dec_list)

for dec in dec_list:
    make_map_and_FFT(dec, 45, freqs[fidx], wcs2, RA_extent=50, dec_extent=20)

In [ ]:
def make_map_and_FFT_v2(dec, RA, freq, wcs, RA_extent=358, dec_extent=20):

    print(dec, freq)

    # RA and dec limit indices:
    RAmin = RA - RA_extent/2
    RAmax = RA + RA_extent/2
    decmin = dec - dec_extent/2
    decmax = dec + dec_extent/2
    idxRA  = np.where( (RAs > RAmin) & (RAs < RAmax) )[0]
    idxdec = np.where( (decs > decmin) & (decs < decmax) )[0]

    # Make subset of map for FFT:
    map_sub = map_new_full[idxdec][:,idxRA]
    print(map_sub.shape)

    # Calculate FFT:
    image_FFT = np.fft.fft2(map_sub)
    image_FFT_shift = np.fft.fftshift(image_FFT)
    uv_m, uv_n = get_uv_axes(map_sub, dxy, freq)

    # Make the plots:
    fig = plt.figure(figsize=(12,11))
    plt.subplots_adjust(left=0.05,right=0.95,top=0.95,bottom=0.1,hspace=-0.6)

    # The map:
    #ax1 = fig.add_subplot(511, projection=wcs.celestial)
    ax1 = fig.add_subplot(511)
    crange = SkyCoord([RAmax,RAmin], [decmin,decmax], frame=ICRS, unit=(u.deg, u.deg))
    #ax1.imshow(map_new_full,origin='lower',vmin=-20,vmax=20,cmap='RdBu_r')
    ax1.imshow(map_sub,origin='lower',vmin=-20,vmax=20,cmap='RdBu_r')
    #ax1.set_ylim(wcs.world_to_pixel(crange)[1])
    #ax1.set_xlim(wcs.world_to_pixel(crange)[0])
    ax1.set_xlabel('RA')
    ax1.set_ylabel('dec')
    ax1.set_title('central dec = '+str(dec)+r'$^{\circ}$')

    # The FFT:
    ax2 = fig.add_subplot(212)
    ax2.imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=30000,cmap='viridis',
                  extent=(uv_m[0],uv_m[-1],uv_n[0],uv_n[-1]))
    ax2.set_xlabel('u (m)')
    ax2.set_ylabel('v (m)')
    ax2.set_xlim(-100,100)
    ax2.set_ylim(-80,80)
    for x in [22,44,66]:
        ax2.axvline(x=x,color='red',linestyle='dashed')
    
    plt.savefig('../PLOTS/uv_dec_'+str(int(dec*10))+'_v3.png')

    return

In [ ]:
dec_list = np.arange(10,71,0.5)
print(dec_list)

for dec in dec_list[0:1]:
    make_map_and_FFT_v2(dec, 180, freqs[fidx], wcs2, RA_extent=358, dec_extent=20)

In [ ]:
def make_map_and_FFT_v3(dec, RA, freq, wcs, RA_extent=358, dec_extent=20):

    print(dec, freq)

    # RA and dec limit indices:
    RAmin = RA - RA_extent/2
    RAmax = RA + RA_extent/2
    decmin = dec - dec_extent/2
    decmax = dec + dec_extent/2
    idxRA  = np.where( (RAs > RAmin) & (RAs < RAmax) )[0]
    idxdec = np.where( (decs > decmin) & (decs < decmax) )[0]

    # Make subset of map for FFT:
    map_sub = map_new_full[idxdec][:,idxRA]
    print(map_sub.shape)

    # Calculate FFT:
    image_FFT = np.fft.fft2(map_sub)
    image_FFT_shift = np.fft.fftshift(image_FFT)
    uv_m, uv_n = get_uv_axes(map_sub, dxy, freq)

    # Make the plots:
    fig = plt.figure(figsize=(12,8))
    plt.subplots_adjust(left=0.05,right=0.95,top=0.95,bottom=0.1,hspace=-0.1)

    # The map:
    ax1 = fig.add_subplot(211, projection=wcs.celestial)
    #ax1 = fig.add_subplot(211)
    crange = SkyCoord([RAmax,RAmin], [decmin,decmax], frame=ICRS, unit=(u.deg, u.deg))
    ax1.imshow(map_new_full,origin='lower',vmin=-20,vmax=20,cmap='RdBu_r')
    #ax1.imshow(map_sub,origin='lower',vmin=-20,vmax=20,cmap='RdBu_r')
    ax1.set_ylim(wcs.world_to_pixel(crange)[1])
    ax1.set_xlim(wcs.world_to_pixel(crange)[0])
    ax1.set_xlabel('RA')
    ax1.set_ylabel('dec')
    ax1.set_title('central dec = '+str(dec)+r'$^{\circ}$')

    # The FFT:
    ax3 = fig.add_subplot(212)
    ax3.plot(uv_m, abs(image_FFT_shift[300,:]),color='k')
    ax3.set_yscale('log')
    ax3.set_ylim(0.1,1e7)
    ax3.set_xlim(-120,120)
    #for x in [-66,-44,-22,22,44,66]:
    #    ax3.axvline(x=x,color='red',linestyle='dashed')

    ax3.axvline(x=22,color='red',linestyle='dashed',label='no factor of 2')
    ax3.axvline(x=-22/2,color='blue',linestyle='dashed',label='with factor of 2')

    ax3.axvline(x=44,color='red',linestyle='dashed')
    ax3.axvline(x=-44/2,color='blue',linestyle='dashed')
    ax3.axvline(x=66,color='red',linestyle='dashed')
    ax3.axvline(x=-66/2,color='blue',linestyle='dashed')
    
    ax3.grid()
    ax3.set_xlabel('u (m)')
    ax3.legend()

    print(uv_n[300])
    
    #plt.savefig('../PLOTS/uv_dec_'+str(int(dec*10))+'_v4.png')
    plt.savefig('../PLOTS/uv_dec_full.png')

    return

In [ ]:
dec_list = np.arange(10,71,0.5)
print(dec_list)

for dec in dec_list:
    make_map_and_FFT_v3(dec, 180, freqs[fidx], wcs2, RA_extent=358, dec_extent=20)

In [ ]:
make_map_and_FFT_v3(50, 180, freqs[fidx], wcs2, RA_extent=358, dec_extent=80)